In [1]:
import pandas as pd
import json
import os
import re
from transformers import MarianMTModel, MarianTokenizer
from dotenv import load_dotenv
from huggingface_hub import login
import openpyxl

In [29]:
# source and copy
df_raw_artefacts = pd.read_csv("../data/raw/artifacts_raw.csv", encoding="utf-8-sig")
df_raw_artf_img = pd.read_csv("../data/raw/artifact_images_raw.csv", encoding="utf-8-sig")

df_artefacts = df_raw_artefacts.copy()
df_artefacts["langue"] = "ar"

df_img_pdf = df_raw_artf_img.copy()
df_img_pdf= df_img_pdf.drop_duplicates(subset="image_name")


In [30]:
# clean object column (regex)
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x if x else None

for col in df_artefacts.columns:
    if df_artefacts[col].dtype == "object":
        df_artefacts[col] = df_clean[col].apply(clean_text)

# null management
df_artefacts = df_artefacts.replace({"": None, "null": None, "None": None})

In [31]:
# json cleaning
df_artefacts["images_associees"] = df_artefacts["images_associees"].apply(
    lambda x: json.loads(x) if pd.notna(x) else []
)

In [32]:
# pictures folder
folder_path = "../figures/pdf_images"

extensions = (".png")

# list
files = [f for f in os.listdir(folder_path) if f.lower().endswith(extensions)]

df_img_folder = pd.DataFrame(files, columns=["image_name"])
df_images = df_img_folder.merge(df_img_pdf, how="left", on="image_name")

In [43]:
def extract_http_link(row):
    for value in row:
        if value is None:
            continue

        text = str(value).strip()

        if text.startswith(("http://", "https://")):
            return text

    return None

df_artefacts["lien_ok"] = df_artefacts.apply(extract_http_link, axis=1)

df_artefacts = df_artefacts.drop(columns=["images_associees", "langue", "lien"], errors="ignore")
#df_artefacts

In [33]:
df_artefacts.to_csv("../data/clean/artefacts_clean.csv", index=False, encoding="utf-8-sig")
df_images.to_csv("../data/clean/images_clean.csv", index=False, encoding="utf-8-sig")

In [38]:
df_artefacts.to_csv("../data/clean/artefacts_clean_final.csv", index=False, encoding="utf-8-sig")

In [44]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import MarianMTModel, MarianTokenizer

# ------------------------
# SETUP
# ------------------------
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

model_name = "Helsinki-NLP/opus-mt-ar-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Device:", device)

# tqdm sur apply
tqdm.pandas()

# ------------------------
# UTILS
# ------------------------
def contains_arabic(text):
    text = str(text)
    return bool(re.search(r"[\u0600-\u06FF]", text))

def smart_translate(text, max_length=256):
    text = "" if pd.isna(text) else str(text).strip()

    if not text:
        return ""

    # si pas d'arabe, on garde tel quel
    if not contains_arabic(text):
        return text

    try:
        with torch.no_grad():
            tokens = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_length
            ).to(device)

            generated = model.generate(**tokens, max_length=max_length)
            result = tokenizer.decode(generated[0], skip_special_tokens=True)

        return result

    except Exception as e:
        print(f"Erreur traduction pour: {text[:80]}... -> {e}")
        return text

# ------------------------
# FONCTION SIMPLE : 1 COLONNE
# ------------------------
def translate_one_column(df, col_name):
    new_col = f"{col_name}_en"

    df[new_col] = (
        df[col_name]
        .fillna("")
        .astype(str)
        .progress_apply(smart_translate)
    )

    print(f"✅ Colonne créée : {new_col}")
    return df

df_artefacts = translate_one_column(df_artefacts, "type")
df_artefacts.to_csv("backup_type.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "description_texte_long")
df_artefacts.to_csv("backup_description_texte_long.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "materiel")
df_artefacts.to_csv("backup_materiel.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "periode")
df_artefacts.to_csv("backup_periode.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "dimensions")
df_artefacts.to_csv("backup_dimensions.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "nom_musee_auction")
df_artefacts.to_csv("backup_nom_musee_auction.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "etat")
df_artefacts.to_csv("backup_etat.csv", index=False, encoding="utf-8-sig")

df_artefacts = translate_one_column(df_artefacts, "note")
df_artefacts.to_csv("backup_note.csv", index=False, encoding="utf-8-sig")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Device: cpu


  0%|          | 0/1125 [00:00<?, ?it/s]

✅ Colonne créée : type_en


...

✅ Colonne créée : nom_musee_auction_en


  0%|          | 0/1125 [00:00<?, ?it/s]

✅ Colonne créée : etat_en


  0%|          | 0/1125 [00:00<?, ?it/s]

✅ Colonne créée : note_en


In [46]:
df_artefacts.to_csv("artefacts_translated_final.csv", index=False, encoding="utf-8-sig")


In [ ]:
df_trans = pd.read_csv("artefacts_translated_final.csv", encoding="utf-8")

cols_for_text = [
    "type_en",
    "description_texte_long_en",
    "materiel_en",
    "periode_en",
    "dimensions_en",
    "nom_musee_auction_en",
    "etat_en",
    "note_en"
]

cols_for_text = [c for c in cols_for_text if c in df.columns]
print("Colonnes utilisées :", cols_for_text)

import re

def clean_for_matching(text):
    text = "" if pd.isna(text) else str(text)

    # supprimer URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # supprimer arabe
    text = re.sub(r"[\u0600-\u06FF]+", " ", text)

    # nettoyer ponctuation bizarre
    text = re.sub(r"[^A-Za-z0-9\s\-.,;:()]", " ", text)

    # espaces propres
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["text_for_matching"] = (
    df[cols_for_text]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .apply(clean_for_matching)
)

In [51]:
df = pd.read_csv("artefacts_translated_final.csv", sep=None, engine="python", encoding="utf-8")

In [65]:
# concatenate all the translated text in one column
import pandas as pd
import re

# Charger le CSV proprement
df = pd.read_csv(
    "artefacts_translated_final.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

# Nettoyer les noms de colonnes
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

# Supprimer les colonnes parasites "Unnamed"
df = df.loc[:, ~df.columns.str.contains("^Unnamed", regex=True)]

# Construire la colonne unique en concaténant tout sauf artifact_id
df["text_for_matching"] = (
    df[[c for c in df.columns if c != "artifact_id"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .apply(
        lambda text: re.sub(
            r"\s+",
            " ",
            re.sub(
                r"[\u0600-\u06FF]+",
                " ",
                re.sub(r"https?://\S+|www\.\S+", " ", text),
            ),
        ).strip()
    )
)

# Sauvegarder
df.to_csv("artefacts_text_only.csv", index=False, encoding="utf-8-sig")

# Vérifier
print(df.columns.tolist())
print(df[["artifact_id", "text_for_matching"]].head())


# i did myself a join on exlce because my eyes are not seeing anything

['artifact_id', 'type_en', 'description_texte_long_en', 'materiel_en', 'periode_en', 'dimensions_en', 'nom_musee_auction_en', 'etat_en', 'note_en', 'text_for_matching']
                          artifact_id  \
0  0-آثارنا المنهوبة_18_p001_593cd272   
1  0-آثارنا المنهوبة_18_p001_669d467d   
2  0-آثارنا المنهوبة_18_p002_6dd48f0e   
3  0-آثارنا المنهوبة_18_p002_ae943cdb   
4  0-آثارنا المنهوبة_18_p003_84275417   

                                   text_for_matching  
0  A sword. A blade of bronze on his handle, writ...  
1  A human statue. A human statue of limestone in...  
2  . A human statue. A human statue from the marb...  
3  A bottle. A bottle of high-transparency bronze...  
4  An earring. A half-circular-shaped gold earrin...  


In [73]:
import pandas as pd

df_new = pd.read_csv(
    "../data/clean/artefacts_parse.csv",
    sep=";",
    engine="python",
    encoding="utf-8-sig"
)

print(df_new.shape)
print(df_new.columns.tolist())
df_new.head()

(1125, 14)
['artifact_id', 'pdf_name', 'page_num', 'numero_inventaire', 'type', 'description_texte_long', 'materiel', 'periode', 'dimensions', 'nom_musee_auction', 'etat', 'note', 'lien_ok', 'text_for_matching']


,artifact_id,pdf_name,page_num,numero_inventaire,type,description_texte_long,materiel,periode,dimensions,nom_musee_auction,etat,note,lien_ok,text_for_matching
0,0-آثارنا المنهوبة_18_p001_593cd272,0-آثارنا المنهوبة_18.pdf,1,NaN,سيف,سيف من مادة البرونز على مقبضه كتابة بخط المسند...,البرونز,NaN,NaN,سوق حزم الجوف,منهوب,NaN,NaN,"A sword. A blade of bronze on his handle, writ..."
1,0-آثارنا المنهوبة_18_p001_669d467d,0-آثارنا المنهوبة_18.pdf,1,NaN,تمثال آدمي,تمثال آدمي من حجر الكلس بوضعية الجلوس واليدين ...,حجر الكلس,NaN,NaN,سوق حزم الجوف,منهوب,باإلضافة إلى خاتم من البرونز,NaN,A human statue. A human statue of limestone in...
2,0-آثارنا المنهوبة_18_p002_6dd48f0e,0-آثارنا المنهوبة_18.pdf,2,NaN,تمثال آدمي,تمثال آدمي من حجر المرمر (الرخام)، بوضعية الوق...,حجر المرمر (الرخام),NaN,NaN,المعروضة للبيع في سوق حزم الجوف,منهوبة,عليها أحرف بخط المسند.,NaN,. A human statue. A human statue from the marb...
3,0-آثارنا المنهوبة_18_p002_ae943cdb,0-آثارنا المنهوبة_18.pdf,2,NaN,قنينة,قنينة من البرونز ذات درجة شفافية عالية، أسفلها...,البرونز,NaN,NaN,المعروضة للبيع في سوق حزم الجوف,منهوبة,NaN,NaN,A bottle. A bottle of high-transparency bronze...
4,0-آثارنا المنهوبة_18_p003_84275417,0-آثارنا المنهوبة_18.pdf,3,NaN,قرط,قرط من الذهب ذو شكل نصف دائري، محاط بصفين من ا...,الذهب,NaN,NaN,سوق حزم الجوف,منهوب من المواقع الأثرية، معروض للبيع,NaN,NaN,An earring. A half-circular-shaped gold earrin...


In [76]:
import pandas as pd
import re

# 1. Charger CSV (Excel → souvent ;)
df = pd.read_csv(
    "../data/clean/artefacts.csv",
    sep=";",
    engine="python",
    encoding="utf-8-sig"
)

# 2. Nettoyer noms de colonnes (BOM + espaces)
df.columns = df.columns.str.replace("\ufeff", "", regex=False).str.strip()

# 3. Supprimer colonnes inutiles type Unnamed
df = df.loc[:, ~df.columns.str.contains("^Unnamed", regex=True)]

# 4. Construire colonne texte unique (tout sauf artifact_id)
df["text_for_matching"] = (
    df[[c for c in df.columns if c != "artifact_id"]]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# 5. Nettoyage texte
def clean_text(text):
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"[\u0600-\u06FF]+", " ", text)        # arabe
    text = re.sub(r"\s+", " ", text).strip()             # espaces propres
    return text

df["text_for_matching"] = df["text_for_matching"].apply(clean_text)

# 6. (optionnel) enlever répétitions consécutives
def remove_consecutive_duplicates(text):
    words = text.split()
    if not words:
        return ""
    cleaned = [words[0]]
    for w in words[1:]:
        if w.lower() != cleaned[-1].lower():
            cleaned.append(w)
    return " ".join(cleaned)

df["text_for_matching"] = df["text_for_matching"].apply(remove_consecutive_duplicates)

# 7. Sauvegarder
df.to_csv("../data/clean/artefacts_parse_clean.csv", index=False, encoding="utf-8-sig")

# 8. Vérifier
print(df[["artifact_id", "text_for_matching"]].head())

                          artifact_id  \
0  0-آثارنا المنهوبة_18_p001_593cd272   
1  0-آثارنا المنهوبة_18_p001_669d467d   
2  0-آثارنا المنهوبة_18_p002_6dd48f0e   
3  0-آثارنا المنهوبة_18_p002_ae943cdb   
4  0-آثارنا المنهوبة_18_p003_84275417   

                                   text_for_matching  
0  0- _18.pdf 1 A sword. A blade of bronze on his...  
1  0- _18.pdf 1 A human statue. A human statue of...  
2  0- _18.pdf 2 ( ) . ( ) . A human statue. A hum...  
3  0- _18.pdf 2 . A bottle. A bottle of high-tran...  
4  0- _18.pdf 3 . An earring. A half-circular-sha...  


In [13]:
"""import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

tqdm.pandas()

# charger ton fichier
df = pd.read_csv("../data/clean/weblink_text.csv", sep=";", encoding="utf-8-sig")

# fonction scrape
def scrape_text(url):
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text(" ", strip=True)

        # nettoyage
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        return text
    except:
        return ""

# appliquer AVEC progression
df["text_for_matching_web"] = df["lien_ok"].progress_apply(scrape_text)

# sauvegarde
df.to_csv("../data/clean/artefacts_with_web.csv", index=False, encoding="utf-8-sig")
"""


<>:20: SyntaxWarning: invalid escape sequence '\S'
<>:20: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_28504\1989936950.py:20: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)


'import pandas as pd\nimport requests\nimport re\nfrom bs4 import BeautifulSoup\nfrom tqdm.auto import tqdm\n\ntqdm.pandas()\n\n# charger ton fichier\ndf = pd.read_csv("../data/clean/weblink_text.csv", sep=";", encoding="utf-8-sig")\n\n# fonction scrape\ndef scrape_text(url):\n    try:\n        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})\n        soup = BeautifulSoup(r.text, "html.parser")\n        text = soup.get_text(" ", strip=True)\n\n        # nettoyage\n        text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)\n        text = re.sub(r"\\s+", " ", text).strip()\n\n        return text\n    except:\n        return ""\n\n# appliquer AVEC progression\ndf["text_for_matching_web"] = df["lien_ok"].progress_apply(scrape_text)\n\n# sauvegarde\ndf.to_csv("../data/clean/artefacts_with_web.csv", index=False, encoding="utf-8-sig")\n'

In [15]:
import pandas as pd
import requests
import re
import html
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from tqdm.auto import tqdm

tqdm.pandas()

# =========================
# CONFIG
# =========================
INPUT_PATH = "../data/clean/weblink_text.csv"
OUTPUT_PATH = "../data/clean/artefacts_with_web.csv"
INPUT_SEP = ";"
TIMEOUT = 20
MAX_OUTPUT_CHARS = 5000

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

BAD_TAGS = [
    "script", "style", "noscript", "svg", "img", "header", "footer",
    "nav", "aside", "form", "button", "input", "iframe"
]

BAD_TEXT_PATTERNS = re.compile(
    r"cookie|privacy|newsletter|subscribe|sign in|login|register|"
    r"facebook|instagram|twitter|linkedin|youtube|tiktok|"
    r"all rights reserved|accept cookies|manage preferences|"
    r"breadcrumb|skip to content|terms of use",
    flags=re.IGNORECASE
)

GOOD_KEYWORDS = re.compile(
    r"artifact|object|statue|head|bust|figure|figurine|relief|stela|stele|"
    r"bowl|vessel|jar|amphora|bronze|marble|limestone|terracotta|ceramic|"
    r"roman|greek|egyptian|byzantine|hellenistic|century|dated|period|"
    r"provenance|dimensions|height|width|depth|cm|inches|lot",
    flags=re.IGNORECASE
)

# =========================
# HELPERS
# =========================
def normalize_text(text):
    if text is None:
        return ""
    text = html.unescape(str(text))
    text = text.replace("\xa0", " ")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def looks_like_javascript_page(text):
    text = (text or "").lower()
    signals = [
        "enable javascript",
        "javascript is required",
        "please turn javascript on",
        "app-root",
        "__next",
        "window.__",
    ]
    return any(s in text for s in signals)

def remove_bad_nodes(soup):
    for tag_name in BAD_TAGS:
        for node in soup.find_all(tag_name):
            node.decompose()

def score_block(tag):
    text = normalize_text(tag.get_text(" ", strip=True))
    if not text:
        return -10_000

    text_len = len(text)

    if text_len < 80:
        return -1000

    p_count = len(tag.find_all("p"))
    li_count = len(tag.find_all("li"))
    heading_count = len(tag.find_all(["h1", "h2", "h3"]))
    keyword_hits = len(GOOD_KEYWORDS.findall(text[:3000]))
    bad_hits = len(BAD_TEXT_PATTERNS.findall(text[:1500]))

    # pénalités si trop "chrome" de page
    penalty = 0
    if bad_hits > 0:
        penalty += 300 * bad_hits

    # score simple mais efficace
    score = (
        text_len
        + p_count * 120
        + li_count * 30
        + heading_count * 40
        + keyword_hits * 100
        - penalty
    )

    return score

def split_into_chunks(text):
    text = normalize_text(text)
    if not text:
        return []

    # coupe en pseudo-phrases / blocs
    chunks = re.split(r"(?<=[\.\!\?])\s+|\s{2,}", text)
    cleaned = []

    seen = set()
    for chunk in chunks:
        chunk = normalize_text(chunk)
        if len(chunk) < 40:
            continue
        if BAD_TEXT_PATTERNS.search(chunk):
            continue
        if chunk in seen:
            continue
        seen.add(chunk)
        cleaned.append(chunk)

    return cleaned

def post_clean(text):
    text = normalize_text(text)

    # enlève quelques boilerplates fréquents
    text = re.sub(r"\bLOT\s*-\s*ART\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bshare\b|\bprint\b|\bdownload\b", " ", text, flags=re.IGNORECASE)

    # normalise espaces
    text = re.sub(r"\s+", " ", text).strip()

    return text[:MAX_OUTPUT_CHARS]

def extract_candidate_blocks(soup):
    selectors = [
        "article",
        "main",
        '[role="main"]',
        ".content",
        ".main-content",
        ".post-content",
        ".entry-content",
        ".object-description",
        ".lot-description",
        ".product-description",
        ".description",
        ".field-name-body",
        ".article-body",
        ".record-description",
        ".views-field-body",
        "#content",
        "#main",
    ]

    candidates = []

    for sel in selectors:
        for node in soup.select(sel):
            candidates.append(node)

    if not candidates:
        for node in soup.find_all(["section", "div"], limit=500):
            candidates.append(node)

    return candidates

def extract_best_text_from_html(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    remove_bad_nodes(soup)

    candidates = extract_candidate_blocks(soup)

    scored = []
    for tag in candidates:
        score = score_block(tag)
        if score > 0:
            scored.append((score, tag))

    if not scored:
        fallback = normalize_text(soup.get_text(" ", strip=True))
        return post_clean(fallback)

    scored.sort(key=lambda x: x[0], reverse=True)

    top_texts = []
    seen = set()

    # on garde les 3 meilleurs blocs uniques
    for score, tag in scored[:20]:
        txt = normalize_text(tag.get_text(" ", strip=True))
        if not txt:
            continue
        if txt in seen:
            continue
        seen.add(txt)
        top_texts.append(txt)
        if len(top_texts) == 3:
            break

    combined = " ".join(top_texts)
    chunks = split_into_chunks(combined)

    # garde les meilleurs chunks
    ranked_chunks = []
    for chunk in chunks:
        chunk_score = len(chunk) + len(GOOD_KEYWORDS.findall(chunk)) * 80
        ranked_chunks.append((chunk_score, chunk))

    ranked_chunks.sort(reverse=True, key=lambda x: x[0])

    final_chunks = []
    total_len = 0
    seen = set()

    for _, chunk in ranked_chunks:
        if chunk in seen:
            continue
        if total_len + len(chunk) > MAX_OUTPUT_CHARS:
            continue
        seen.add(chunk)
        final_chunks.append(chunk)
        total_len += len(chunk) + 1

        # garde environ 6 à 10 bons blocs max
        if len(final_chunks) >= 8:
            break

    if not final_chunks:
        return post_clean(combined)

    return post_clean(" ".join(final_chunks))

def domain_specific_extract(soup, domain):
    domain = domain.lower()

    selectors_by_domain = {
        "christies.com": [
            ".lotdescription",
            ".description",
            "main",
        ],
        "sothebys.com": [
            ".lot-detail-description",
            ".css-*",
            "main",
        ],
        "metmuseum.org": [
            ".artwork__intro",
            ".artwork__content",
            "main",
        ],
        "britishmuseum.org": [
            ".article__body",
            ".content",
            "main",
        ],
    }

    for known_domain, selectors in selectors_by_domain.items():
        if known_domain in domain:
            for sel in selectors:
                try:
                    nodes = soup.select(sel)
                    if nodes:
                        text = " ".join(
                            normalize_text(n.get_text(" ", strip=True))
                            for n in nodes
                        )
                        text = post_clean(text)
                        if len(text) > 120:
                            return text
                except Exception:
                    pass

    return ""

def scrape_text(url):
    if pd.isna(url) or not str(url).strip():
        return ""

    url = str(url).strip()

    try:
        response = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        response.raise_for_status()
    except Exception:
        return ""

    raw_html = response.text
    if not raw_html:
        return ""

    soup = BeautifulSoup(raw_html, "html.parser")
    remove_bad_nodes(soup)

    domain = urlparse(url).netloc.lower()

    # 1) extraction spécifique domaine si possible
    specific = domain_specific_extract(soup, domain)
    if specific and len(specific) > 120:
        return specific

    # 2) extraction générique robuste
    extracted = extract_best_text_from_html(raw_html)

    # 3) fallback si page très JS / très pauvre
    if len(extracted) < 120:
        visible_text = normalize_text(soup.get_text(" ", strip=True))
        if looks_like_javascript_page(visible_text):
            return "[JS_PAGE_OR_POOR_EXTRACTION]"
        return visible_text[:MAX_OUTPUT_CHARS]

    return extracted

# =========================
# LOAD
# =========================
df = pd.read_csv(INPUT_PATH, sep=INPUT_SEP, encoding="utf-8-sig")

# =========================
# SCRAPE
# =========================
df["text_for_matching_web"] = df["lien_ok"].progress_apply(scrape_text)

# =========================
# CHECKS
# =========================
df["web_text_len"] = df["text_for_matching_web"].fillna("").str.len()

print("Shape:", df.shape)
print("\nLongueur text_for_matching_web")
print(df["web_text_len"].describe())

print("\nExemples les plus courts :")
display(df.loc[:, ["artifact_id", "lien_ok", "text_for_matching_web", "web_text_len"]].sort_values("web_text_len").head(5))

print("\nExemples les plus longs :")
display(df.loc[:, ["artifact_id", "lien_ok", "text_for_matching_web", "web_text_len"]].sort_values("web_text_len", ascending=False).head(5))

# =========================
# SAVE
# =========================
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\nFichier sauvegardé : {OUTPUT_PATH}")

100%|██████████| 685/685 [15:26<00:00,  1.35s/it]  

Shape: (685, 5)

Longueur text_for_matching_web
count     685.000000
mean      543.575182
std       948.008406
min         0.000000
25%         0.000000
50%         0.000000
75%       947.000000
max      4999.000000
Name: web_text_len, dtype: float64

Exemples les plus courts :


,artifact_id,lien_ok,text_for_matching_web,web_text_len
4,0-آثارنا المنهوبة_25_p005_a9b6090a,https://www.lot-art.com/auction-lots/Arabian-S...,,0
651,0-آثارنا المنهوبة_15_p008_6bee6ec6,https://www.lotart.com/auction/03.10.23-south-...,,0
65,0-آثارنا المنهوبة_25_p066_7c445bf9,https://www.liveauctioneers.com/search/?keywor...,,0
682,0-آثارنا المنهوبة_15_p046_0de7aaca,https://www.ancient-agate-jasper-carnelian-asi...,,0
387,31-آثارنا المنهوبة_31_p037_20e4ce58,https://www.sothebys.com/en/buy/auction/2025/i...,,0



Exemples les plus longs :


,artifact_id,lien_ok,text_for_matching_web,web_text_len
609,0-آثارنا المنهوبة_22_p022_e2eb1dc8,https://www.bonhams.com/auctions/21928/lot/129/,How to sell 269 — Five Roman glass vessels 5 £...,4999
573,0-آثارنا المنهوبة 30_p009_8f0c48a1,https://www.bonhams.com/auction/20668/lot/181/...,How to sell 269 — Five Roman glass vessels 5 £...,4987
572,0-آثارنا المنهوبة 30_p008_6098ac5e,https://www.bonhams.com/auction/30769/lot/73/s...,African and Oceanic Art Antiquities Home and I...,4985
582,0-آثارنا المنهوبة 30_p018_880a3521,https://www.metmuseum.org/art/collection/searc...,Incense burner ca. mid-1st millennium BCE Not ...,4980
606,0-آثارنا المنهوبة_22_p019_3b93fef1,https://www.bonhams.com/auctions/20467/lot/201/,How to sell 269 — Five Roman glass vessels 5 £...,4972



Fichier sauvegardé : ../data/clean/artefacts_with_web.csv


In [18]:
# Script to clean images folder
import os
import hashlib

# dossier contenant les images extraites
output_dir_img = "../figures/web_photos/"

def hash_file(path):
    """Renvoie le hash MD5 du fichier"""
    h = hashlib.md5()
    with open(path, "rb") as f:
        h.update(f.read())
    return h.hexdigest()

def remove_duplicate_images(folder):
    seen_hashes = set()
    removed_files = []
    
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        
        # ignorer les sous-dossiers
        if not os.path.isfile(img_path):
            continue
        
        try:
            h = hash_file(img_path)
        except PermissionError:
            print(f"⚠️ Permission refusée pour {img_path}, passage...")
            continue
        
        if h in seen_hashes:
            os.remove(img_path)
            removed_files.append(filename)
        else:
            seen_hashes.add(h)
    
    print(f"✅ Doublons supprimés : {len(removed_files)}")
    if removed_files:
        print("Fichiers supprimés :", removed_files)

# Exécution
remove_duplicate_images(output_dir_img)

✅ Doublons supprimés : 169
Fichiers supprimés : ['0-_30_p009_8f0c48a1_img2_2faf821d9e_03aec49d9e57.jpg', '0-_30_p009_8f0c48a1_img3_d95995a197_a392a6275003.jpg', '0-_30_p020_5b4e7eb9_img3_66a47059f7_0c5dfe495ea0.jpg', '0-_30_p021_65971c7e_img3_66a47059f7_0c5dfe495ea0.jpg', '0-__10_p095_f45e4544_img2_2faf821d9e_03aec49d9e57.jpg', '0-__10_p095_f45e4544_img3_d95995a197_a392a6275003.jpg', '0-__15_p005_673dba90_img1_9090cef453_cc77d6620469.jpg', '0-__15_p006_52386031_img1_9090cef453_cc77d6620469.jpg', '0-__15_p007_def7f6ce_img1_9090cef453_cc77d6620469.jpg', '0-__15_p009_916ab2dc_img1_9090cef453_cc77d6620469.jpg', '0-__16_p010_1a66044c_img2_542e7efebf_883abb502cef.jpg', '0-__16_p012_54866ff3_img1_a17e2cc683_40d56a10516a.jpg', '0-__16_p012_54866ff3_img2_02a1d8a9ab_a29ff51e1db1.jpg', '0-__16_p014_9c48a41b_img1_02a1d8a9ab_a29ff51e1db1.jpg', '0-__16_p014_9c48a41b_img2_a17e2cc683_40d56a10516a.jpg', '0-__19_p009_7128eb36_img1_851e4233e9_755b4416d78c.jpg', '0-__19_p010_da2683d3_img1_851e4233e9_755b4

In [95]:

# prepare data for relational db

# web link

df = pd.read_csv("../data/clean/web_links.csv", sep = ";", encoding="utf-8-sig")
#df = df.reset_index(drop=True)
#df["id_page"] = df.index+1

# artefacts
#df_art = pd.read_csv("../data/clean/looted_artefacts.csv", sep = ";", encoding="utf-8-sig")
#df_merge_art = df_art.merge(df, how = "left", on = "artifact_id")
#df_merge_art.drop(columns=['lien_ok_x','lien_ok_y'], inplace = True)

# web images
df_web_photos = pd.read_csv("../data/clean/web_photos.csv", sep = ",", encoding="utf-8-sig")
#df_merge_wb_ph = df_web_photos.merge(df, how = "left", on = "artifact_id")
#df_merge_wb_ph.drop('lien_ok', axis = 1, inplace = True)

# web texts
#df_web_text = pd.read_csv("../data/clean/web_text.csv", sep = ";", encoding="utf-8-sig")
#df_merge_txt = df_web_text.merge(df, how = "left", on = "artifact_id")
#df_merge_txt.drop(columns=['lien_ok_x','lien_ok_y', 'text_for_matching', 'web_text_len'], inplace = True)
#df_merge_txt.to_csv("web_text.csv", encoding="utf-8-sig") 

#df_web_photos.drop('Unnamed: 0', axis = 1, inplace = True)
#filtered_df_web_photos = df_web_photos[df_web_photos["status"] == "downloaded"]
#filtered_df_web_photos.drop('status', axis = 1, inplace = True)

# web photos folder
folder_path = "../figures/web_photos"

extensions = ".jpg", ".webp"

# list
files = [f for f in os.listdir(folder_path) if f.lower().endswith(extensions)]

# keep only pictures from the folder in order to create a df consistent between files and dataset
#df_img_folder = pd.DataFrame(files, columns=["image_name"])
#df_web_photos = df_img_folder.merge(filtered_df_web_photos, left_on="image_name", right_on="local_filename", how="inner")
#df_web_photos.drop('Unnamed: 0', axis = 1, inplace = True)
#df_web_photos.to_csv("web_photos.csv", encoding="utf-8-sig") 
#df_web_photos.columns


Index(['image_name', 'artifact_id', 'source_page_url', 'image_rank',
       'image_url', 'local_filename', 'width', 'height', 'id_page'],
      dtype='str')

In [101]:
from dotenv import load_dotenv
import os
import json
import time
import re
import pandas as pd
from openai import OpenAI

load_dotenv()

USE_OPENAI = True
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

INPUT_CSV = "../data/raw/data_versions/prepa_text.csv"
OUTPUT_CSV = "../data/raw/data_versions/output_rewritten.csv"

COLUMN_1 = "text_for_matching_web"
COLUMN_2 = "text_for_matching"

OUTPUT_PRICE_COL = "price"
OUTPUT_CURRENCY_COL = "currency"
OUTPUT_SHORT_COL = "description_short"
OUTPUT_LONG_COL = "description_long"
OUTPUT_COL2_DESC = "description_col2"

REQUEST_DELAY_SECONDS = 1
MAX_RETRIES = 3

if USE_OPENAI and not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing from environment variables.")

client = OpenAI(api_key=OPENAI_API_KEY)

SYSTEM_PROMPT_MAIN = """You are an expert in writing archaeological object records for museum databases.

Your task is to transform the provided text into a structured, detailed, neutral description.

STRICT RULES:
- Always begin with the object type, followed by the main material.
- Keep only factual information about the object.
- Remove commercial mentions.
- Do not invent anything.
- Do not infer beyond the source text.
- Preserve technical vocabulary (materials, techniques, typology, shape, dimensions, period).
- Rewrite in clear and fluent English.
- Produce dense and informative descriptions.

The output must contain:
- a price field
- a currency field
- a short paragraph
- a long paragraph

CONTENT RULES:
- The short paragraph must be concise, factual, and written in one paragraph.
- The long paragraph must be more detailed, still factual, and written in one paragraph.
- If price or currency is not explicitly present, return an empty string for that field.
- If some descriptive information is missing, do not invent it.

OUTPUT FORMAT:
Return only valid JSON with exactly these keys:
price
currency
description_short
description_long
"""

SYSTEM_PROMPT_COL2 = """You are an expert in writing archaeological object records for museum databases.

Your task is to transform the provided text into a structured, detailed, neutral description.

STRICT RULES:
- Always begin with the object type, followed by the main material.
- Keep only factual information about the object.
- Remove commercial mentions.
- Do not invent anything.
- Do not infer beyond the source text.
- Preserve technical vocabulary (materials, techniques, typology, shape, dimensions, period).
- Rewrite in clear and fluent English.
- Produce a dense and informative description.

OUTPUT FORMAT:
Return only plain text.
Return one paragraph only.
Do not return JSON.
"""

def clean_text(value: str) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def call_llm_main(source_text: str) -> dict:
    user_prompt = f"""Source text:
{source_text}

Return only JSON.
"""

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_MAIN},
            {"role": "user", "content": user_prompt},
        ],
    )

    content = response.choices[0].message.content
    data = json.loads(content)

    return {
        OUTPUT_PRICE_COL: clean_text(data.get("price", "")),
        OUTPUT_CURRENCY_COL: clean_text(data.get("currency", "")),
        OUTPUT_SHORT_COL: clean_text(data.get("description_short", "")),
        OUTPUT_LONG_COL: clean_text(data.get("description_long", "")),
    }

def call_llm_col2(source_text: str) -> str:
    user_prompt = f"""Source text:
{source_text}

Return only one paragraph of plain text.
"""

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_COL2},
            {"role": "user", "content": user_prompt},
        ],
    )

    return clean_text(response.choices[0].message.content)

def safe_call_main(source_text: str) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call_llm_main(source_text)
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(REQUEST_DELAY_SECONDS * attempt)
            else:
                return {
                    OUTPUT_PRICE_COL: "",
                    OUTPUT_CURRENCY_COL: "",
                    OUTPUT_SHORT_COL: "",
                    OUTPUT_LONG_COL: "",
                }

def safe_call_col2(source_text: str) -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call_llm_col2(source_text)
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(REQUEST_DELAY_SECONDS * attempt)
            else:
                return ""

def process_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    main_results = []
    col2_results = []

    for _, row in df.iterrows():
        text_1 = clean_text(row.get(COLUMN_1, ""))
        text_2 = clean_text(row.get(COLUMN_2, ""))

        if text_1:
            main_result = safe_call_main(text_1)
        else:
            main_result = {
                OUTPUT_PRICE_COL: "",
                OUTPUT_CURRENCY_COL: "",
                OUTPUT_SHORT_COL: "",
                OUTPUT_LONG_COL: "",
            }

        if text_2:
            col2_result = safe_call_col2(text_2)
        else:
            col2_result = ""

        main_results.append(main_result)
        col2_results.append(col2_result)

        time.sleep(REQUEST_DELAY_SECONDS)

    main_results_df = pd.DataFrame(main_results)
    col2_results_df = pd.DataFrame({OUTPUT_COL2_DESC: col2_results})

    return pd.concat(
        [df.reset_index(drop=True), main_results_df, col2_results_df],
        axis=1
    )

def main() -> None:
    df = pd.read_csv(INPUT_CSV, sep=";", quotechar='"', engine="python")
    df = process_dataframe(df)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

if __name__ == "__main__":
    main()

In [126]:
# pdf_text upgraded

df_pdf_upgrd = pd.read_csv("../data/raw/data_versions/pdf_upgraded_text.csv", sep = ",", encoding="utf-8-sig")
df_pdf_text = pd.read_csv("../data/clean/looted_artefacts.csv", sep = ",", encoding="utf-8-sig")

In [136]:
#df_merge = df_pdf_text.merge(df_pdf_upgrd, on="artifact_id", how="left")
#df_merge.drop('Unnamed: 0', axis = 1, inplace=True)
#df_merge.columns
df_merge.to_csv("../data/clean/looted_artefacts.csv", encoding="utf-8-sig") 


In [138]:
# en attente
df = pd.read_csv("../data/clean/web_upgraded_text.csv", sep = ";", encoding="utf-8-sig")
df = df.reset_index(drop=True)
df["id_article"] = df.index+1
df.to_csv("../data/clean/web_upgraded_text.csv", encoding="utf-8-sig") 

In [158]:
#artefacts = pd.read_csv("../data/clean/looted_artefacts.csv", sep = ",", encoding="utf-8-sig")
#articles = pd.read_csv("../data/clean/web_upgraded_text.csv", sep = ",", encoding="utf-8-sig")
#df_merge =articles.merge(artefacts, on="artifact_id", how="inner")
#df_merge.drop(columns = ['Unnamed: 0_x', 'text_for_matching_web', 'id_page_x',
      # 'price', 'currency', 'id_article', 'Unnamed: 0_y', 'pdf_name', 'page_num',
      # 'numero_inventaire', 'type', 'description_texte_long', 'materiel',
      # 'periode', 'dimensions', 'nom_musee_auction', 'etat', 'note',
      # 'text_for_matching', 'id_page_y'], inplace=True)

df = df_merge.rename(columns={
    "description_short": "wb_txt_shrt",
    "description_long": "wb_txt_lg",
    "upgr_desc": "looted_text"
})

df = df.dropna(subset=["wb_txt_shrt"])

In [159]:
df.to_csv("../data/clean/artifacts_comparison.csv", encoding="utf-8-sig") 

In [162]:
df.artifact_id.unique()

<ArrowStringArray>
['0-آثارنا المنهوبة_25_p020_e27076bd', '0-آثارنا المنهوبة_25_p027_b022fb35',
 '0-آثارنا المنهوبة_25_p036_bea25d9f', '0-آثارنا المنهوبة_25_p037_76bcbec5',
 '0-آثارنا المنهوبة_25_p038_a303509a', '0-آثارنا المنهوبة_25_p039_21a0fd74',
 '0-آثارنا المنهوبة_25_p041_16bc353c', '0-آثارنا المنهوبة_25_p043_e92bb9bc',
 '0-آثارنا المنهوبة_25_p044_1394a6c0', '0-آثارنا المنهوبة_25_p045_eaae26b1',
 ...
 '0-آثارنا المنهوبة_14_p037_d8eb0074', '0-آثارنا المنهوبة_14_p038_e61326bd',
 '0-آثارنا المنهوبة_14_p039_ddf060e7', '0-آثارنا المنهوبة_14_p040_58885c6e',
 '0-آثارنا المنهوبة_14_p041_6240297d', '0-آثارنا المنهوبة_14_p042_78ff3cb5',
 '0-آثارنا المنهوبة_14_p043_1add3b82', '0-آثارنا المنهوبة_14_p044_dfa1c7a5',
 '0-آثارنا المنهوبة_14_p045_fd5a4154', '0-آثارنا المنهوبة_15_p026_22dd2aa6']
Length: 227, dtype: str

In [24]:
# missing add path for next image embedding
import numpy as np
df = pd.read_csv("../data/clean/pdf_images.csv", encoding="utf-8-sig")
directory = r"C:\Users\Utilisateur\Desktop\IRONHACK_DA\COURSES\FINAL_PROJECT\final_project_looted_artefacts\figures\pdf_images"
df['image_path'] = directory + "/" + df['image_name']
df.to_csv("../data/clean/pdf_images.csv", encoding="utf-8-sig")

# another option
#import os
#a = "../data/raw/first_text_scoring.csv" # Relative path
#b = os.path.abspath(a)    # Get absolute path

#print(b)

**Sanitizing data for final relational db**

In [4]:
input_dir = "../data/raw/last_cleaned_data/"
output_dir ="../data/clean/"

In [ ]:
# Looted artefacts (-> add source metadata and final text)
looted_artefacts_sanitized = pd.read_excel(input_dir+"looted_artefacts_sanitized.xlsx")
looted_artefacts = pd.read_csv(input_dir+"looted_artefacts.csv", sep =';', encoding="utf-8-sig")
looted_text = pd.read_csv("../data/raw/4.vector_db/first_text_scoring.csv", sep =';', encoding="utf-8-sig")
final_looted_artefacts = looted_artefacts_sanitized.merge(looted_artefacts[['artifact_id','pdf_name', 'page_num']], on = 'artifact_id', how = 'left')
final_looted_artefacts = final_looted_artefacts.merge(looted_text[['artifact_id','looted_text']], on = 'artifact_id', how = 'left')
final_looted_artefacts.rename(columns={"looted_text": "pdf_upgrd_description"},inplace=True)

In [91]:
# Web related pages (-> add domain of the website)
web_pages = pd.read_csv(input_dir+"web_pages.csv", sep =';', encoding="utf-8-sig")
web_links = pd.read_csv(input_dir+"web_links.csv", sep =';', encoding="utf-8-sig")
artifacts_main_web_sites = pd.read_excel(input_dir+"artifacts_main_web_sites.xlsx")
web_pages_links = web_pages.merge(web_links[['artifact_id','lien_ok']], on = 'artifact_id', how = 'left')
final_web_pages = web_pages_links.merge(artifacts_main_web_sites, on = 'artifact_id', how = 'left')
final_web_pages['article_id'] = final_web_pages.index+1
final_web_pages.head(1)

,artifact_id,text_for_matching_web,id_page,price,currency,wb_txt_shrt,wb_txt_lg,lien_ok,main_website,article_id
0,0-آثارنا المنهوبة_25_p020_e27076bd,Ancient Egyptian Shabti- Ancient Egyptian Amul...,20,NaN,NaN,"Ancient Egyptian antiquities, including shabti...","Ancient Egyptian antiquities, including shabti...",https://artandantiquities.co.uk/a1gallery3.asp...,artandantiquities.co.uk,1


In [ ]:
# Art dealers 
art_dealers = pd.read_excel(input_dir+"website_cities.xlsx")
art_dealers.rename(columns={"input_domain": "main_website"},inplace=True)

In [92]:
# web_photos and pdf_images (-> keep only what is necessary) keep separated because linked to different folders - easier
#web_photos = pd.read_csv(input_dir + "web_photos.csv", sep = ';', encoding="utf-8-sig")
#pdf_images = pd.read_csv(input_dir + "pdf_images.csv", sep = ';', encoding="utf-8-sig")
#web_photos = web_photos[['image_name', 'artifact_id','image_path']]
#final_web_photos = web_photos.merge(final_web_pages[['artifact_id','article_id']],on = 'artifact_id', how = 'left')
#final_pdf_images = pdf_images[['image_name', 'artifact_id','image_path']]

In [101]:
#score = pd.read_csv(input_dir+"final_score.csv", sep =',', encoding="utf-8-sig")
#score['index'] = score.index+1
#score.drop('Unnamed: 0', axis = 1, inplace = True)

In [107]:
# forced to remove arabic text due to mysql import issues
final_looted_artefacts['artifact_id'] = final_looted_artefacts['artifact_id'].str.replace(r'[؀-ۿ]+', '', regex=True)
final_web_pages['artifact_id'] = final_web_pages['artifact_id'].str.replace(r'[؀-ۿ]+', '', regex=True)
final_web_photos['artifact_id'] = final_web_photos['artifact_id'].str.replace(r'[؀-ۿ]+', '', regex=True)
final_pdf_images['artifact_id'] = final_pdf_images['artifact_id'].str.replace(r'[؀-ۿ]+', '', regex=True)
score['artifact_id'] = score['artifact_id'].str.replace(r'[؀-ۿ]+', '', regex=True)

In [111]:
final_looted_artefacts.to_csv(output_dir+"looted_artefacts.csv", sep=',', encoding="utf-8", index=False)
final_web_pages.to_csv(output_dir+"web_pages.csv", sep=',', encoding="utf-8", index=False)
art_dealers.to_csv(output_dir+"art_dealers.csv", sep=',', encoding="utf-8", index=False)
final_web_photos.to_csv(output_dir+"web_photos.csv", sep=',', encoding="utf-8", index=False)
final_pdf_images.to_csv(output_dir+"pdf_images.csv", sep=',', encoding="utf-8", index=False)
score.to_csv(output_dir+"match_scoring.csv", sep=',', encoding="utf-8", index=False)

**Still issues need to regex it again**

In [27]:
# define a common function for all regex issues, expecting it to be ok
import unicodedata
from ftfy import fix_text

def clean_text_and_pray(x):
    if pd.isna(x):
        return ""
    x = fix_text(x) #clean weird chars
    x = str(x) #to add before try
    x = x.replace("Ã—", "×")  
    x = x.replace("â€“", "–") 
    x = x.replace("â€”", "—")
    x = x.replace("â€™", "’")
    x = x.replace("â€˜", "‘")
    x = x.replace("â€œ", "“")
    x = x.replace("â€\x9d", "”")
    x = x.replace("Â ", " ")
    x = x.replace("Â", "")
    try:
       x = x.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    x = unicodedata.normalize("NFKC",x) # normalization unicode
    x = x.replace("\n"," ").replace("\r"," ").replace("\t"," ") # tabs and backlines
    x = x.replace("\u00a0"," ") # - when a text is cut non breaking space
    x = x.replace(";"," ") # - when a text is cut non breaking space
    x = re.sub(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]+','',x) #all arabic chars
    x = re.sub(r"\s+"," ",x).strip()  #normalize to keep only 1 space no use for path might have some space
    x = re.sub(r'[\x00-\x1F\x7F]', '',x)  # control ASCII
    x = re.sub(r'[\u200B-\u200D\uFEFF]', '',x)  # invisible
    x = re.sub(r"0-[_\s]+", "0-", x)
    return x

def clean_path(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = fix_text(x) #clean weird chars
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("\n", "").replace("\r", "").replace("\t", "")
    x = x.replace("\u00a0", " ")
    x = re.sub(r'[\u200B-\u200D\uFEFF]', '', x)
    x = re.sub(r'[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]+', '', x)
    x = re.sub(r"0-[_\s]+", "0-", x)
    x = x.replace("\\", "/")
    x = re.sub(r'/\s+', '/', x)
    x = x.strip()
    x = x.rstrip(" .") 
    return x

In [10]:
# example of functions applied for some columns for data cleaning
wb= pd.read_csv("../data/raw/last_cleaned_data/v1/artifacts_main_web_sites.csv", sep = ';', encoding="utf-8")
cols = ['artifact_id']
for col in cols:
    wb[col] = wb[col].apply(clean_text_and_pray)
wb.to_csv(output_dir+"web_pages_rgx_clean.csv", sep = ';', encoding="utf-8", index=False)
# always index = False, prefer sep =; for mysql

In [16]:
wb_clean= pd.read_csv("../data/clean/csv_to_sql/web_pages_rgx_clean.csv", sep = ';', encoding="utf-8")
wb_p= pd.read_csv("../data/clean/csv_to_sql/web_pages_rgx.csv", sep = ';', encoding="utf-8")
wm = wb_clean.merge(wb_p, on = "artifact_id", how = "left")
wm.article_id.max()

np.float64(240.0)

In [17]:
complete_id = wm["article_id"].isna()
wm.loc[complete_id, "article_id"] = range(int(wm["article_id"].max()) + 1, int(wm["article_id"].max()) + 1 + complete_id.sum())

In [18]:
wm.to_csv(output_dir+"web_pages_rgx_all.csv", sep = ';', encoding="utf-8", index=False)

In [21]:
wb_clean= pd.read_csv("../data/clean/web_pages_rgx_all.csv", sep = ';', encoding="utf-8")

In [24]:
wb_clean.drop('Unnamed: 9', axis = 1, inplace = True)

In [26]:
wb_clean.columns

Index(['artifact_id', 'text_for_matching_web', 'id_page', 'price', 'currency',
       'wb_txt_shrt', 'wb_txt_lg', 'lien_ok', 'main_website', 'article_id'],
      dtype='str')

In [29]:
cols = ['artifact_id', 'text_for_matching_web','price', 'currency',
       'wb_txt_shrt', 'wb_txt_lg', 'lien_ok']
for col in cols:
    wb_clean[col] = wb_clean[col].apply(clean_text_and_pray)
wb_clean.to_csv(output_dir+"web_pages_rgx_1.csv", sep = ';', encoding="utf-8", index=False)

In [33]:

wb_clean["id_page"] = wb_clean["id_page"].fillna(0).astype(int)

In [35]:
wb_clean.to_csv(output_dir+"web_pages_rgx_1.csv", sep = ';', encoding="utf-8", index=False)